In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, max as _max, coalesce, lit
from delta.tables import DeltaTable

# --------------------------------------------------
# Configuration
# --------------------------------------------------
bronze_table = "bronze_dev.global_mart_retail.raw_data"
fact_table   = "silver_dev.global_mart_retail.fact_sales"

dim_customer = "silver_dev.global_mart_retail.dim_customer"
dim_product  = "silver_dev.global_mart_retail.dim_product"
dim_date     = "silver_dev.global_mart_retail.dim_date"

# --------------------------------------------------
# Determine last loaded ingestion_ts for incremental load
# --------------------------------------------------
if spark.catalog.tableExists(fact_table):
    last_loaded_ts = (
        spark.table(fact_table)
        .agg(_max("ingestion_ts").alias("max_ingestion_ts"))
        .collect()[0]["max_ingestion_ts"]
    )
else:
    last_loaded_ts = None

# --------------------------------------------------
# Load Bronze Data Incrementally
# --------------------------------------------------
bronze_df = spark.table(bronze_table)

if last_loaded_ts:
    bronze_df = bronze_df.filter(col("ingestion_ts") > last_loaded_ts)

# --------------------------------------------------
# Join Dimensions (only current records)
# --------------------------------------------------

fact_df = (
    bronze_df.alias("b")
    .join(
        spark.table(dim_customer).alias("c").filter(col("is_current_record") == True),
        col("b.customer_id") == col("c.customer_id"),
        "left"
    )
    .join(
        spark.table(dim_product).alias("p").filter(col("is_current_record") == True),
        col("b.product_id") == col("p.product_id"),
        "left"
    )
    .join(
        spark.table(dim_date).alias("od"),
        to_date(col("b.order_date")) == col("od.date"),
        "left"
    )
    .join(
        spark.table(dim_date).alias("sd"),
        to_date(col("b.ship_date")) == col("sd.date"),
        "left"
    )
    .select(
        # ---- DIMENSION KEYS (CAST AFTER COALESCE) ----
        coalesce(col("c.customer_key"), lit(-1)).cast("int").alias("customer_key"),
        coalesce(col("p.product_key"), lit(-1)).cast("int").alias("product_key"),
        coalesce(col("od.date_key"), lit(19500101)).cast("int").alias("order_date_key"),
        coalesce(col("sd.date_key"), lit(19500101)).cast("int").alias("ship_date_key"),

        # ---- DEGENERATE DIMENSIONS ----
        col("b.order_id").cast("string"),
        col("b.ship_mode").cast("string"),

        # ---- MEASURES ----
        col("b.quantity").cast("int"),
        col("b.sales").cast("decimal(12,2)").alias("sales_amount"),
        col("b.discount").cast("decimal(12,2)").alias("discount_amount"),
        col("b.profit").cast("decimal(12,2)").alias("profit_amount"),

        # ---- AUDIT ----
        col("b.ingestion_ts").cast("timestamp"),
        col("b.source_file_path").cast("string"),
        current_timestamp().cast("timestamp").alias("load_timestamp")
    )
)


before_count = (
    spark.table(fact_table).count()
    if spark.catalog.tableExists(fact_table)
    else 0
)

fact_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(fact_table)

after_count = spark.table(fact_table).count()

inserted_records = after_count - before_count

print(f"Fact Sales Load Completed")
print(f"Records inserted in this run : {inserted_records}")
print(f"Total records in fact table : {after_count}")